In [ ]:
from pathlib import Path, PosixPath
import asyncio
import logging
import re
from urllib.parse import urlparse

from obstore.store import S3Store
import rustac
import duckdb
import s3fs
import os


from pathlib import Path, PosixPath
import logging


logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
import os.path


year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')
paths = []
sizes = []


source = Path("../../catalog/sentinel1//")
destination = Path("geoparquet/sentinel1")

try:
    if not destination.exists():
        destination.mkdir(parents=True, exist_ok=True)
except Exception as e:
    raise RuntimeError(f"Failed to create directory {destination}: {str(e)}")

for path in source.glob("**/*.ndjson"):
    if year_file_re.match(str(path)):
        paths.append(path)
        sizes.append(os.path.getsize(path))

print(len(paths))

In [ ]:
paths[120]

In [ ]:
from asyncio import TaskGroup, Semaphore
import tqdm
import rustac

# Limit the number of files we hold in memory at a time
semaphore = Semaphore(2)
# Store the ones that error, it's the internet after all, things will error
missed_paths = []


async def copy_and_convert(
    source_path: str, destination_path: Path, progress_bar: tqdm.tqdm
) -> None:
    async with semaphore:
        try:
            value = await rustac.read(str(source_path))
            if destination_path.is_dir():
                print(f"✅ '{destination_path}' is a directory")
                missed_paths.append(path)
                progress_bar.update()
                return

        except Exception as e:
            print(e)
            missed_paths.append(path)
            progress_bar.update()
            return

        # ndjson with only one item are read as an item, not a item collection
        try:
            if value["type"] == "Feature":
                await rustac.write(str(destination_path), [value])
            else:
                assert value["type"] == "FeatureCollection"
                await rustac.write(str(destination_path), value)
        except Exception as e:
            print(e)
            missed_paths.append(path)
            progress_bar.update()
            return

        progress_bar.update()


progress_bar = tqdm.tqdm(total=len(paths), miniters=1)
async with TaskGroup() as task_group:
    for path in paths:
        path_parts = path.parts[-2:]  # Get ('N70W030', '2015.ndjson')
        destination_path = destination.joinpath(*path_parts).with_suffix(".parquet")
        base_dir = destination.joinpath(*path.parts[-2:-1])
        await asyncio.to_thread(base_dir.mkdir, parents=True, exist_ok=True)
        # print(destination_path)
        if destination_path.exists():
            progress_bar.update()
        else:
            task_group.create_task(
                copy_and_convert(path, destination_path, progress_bar)
            )

In [ ]:
len(missed_paths)

### Cloud Listing

In [ ]:
# year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')

# source_store = S3Store(
#     bucket="its-live-data", prefix="test-space/cloud-experiments/catalog/sentinel1-consolidated/", region="us-west-2", skip_signature=True
# )

# paths = []
# sizes = []
# for list_stream in source_store.list():
#     for object_meta in list_stream:
#         if year_file_re.match(object_meta["path"]):
#             paths.append(object_meta["path"])
#             sizes.append(object_meta["size"])
# print(len(paths))

### Local Listing

In [ ]:
import os.path


year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')
paths = []
sizes = []

source = Path("../../catalog/sentinel2")
destination = Path("../../catalog/geoparquet/sentinel2")

try:
    if not destination.exists():
        destination.mkdir(parents=True, exist_ok=True)
except Exception as e:
    raise RuntimeError(f"Failed to create directory {destination}: {str(e)}")

for path in source.glob("**/*.ndjson"):
    if year_file_re.match(str(path)):
        paths.append(path)
        sizes.append(os.path.getsize(path))

print(len(paths))

In [ ]:
paths[0]

In [ ]:

results = process_files_batches_per_worker(
    client, 
    paths,
    destination, 
    progress=True
)

In [ ]:
failed = [r["path"] for f in results]
len(failed)

In [ ]:
for r in results:
    if r["status"] != "success":
        print(r)

In [ ]:
import duckdb


destination = Path("./geoparquet")

duckdb.sql(f"select count(*) from read_parquet('{destination}/**/*.parquet')")

In [ ]:
from shapely.geometry import shape, box

feature = {
  "type": "Feature",
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [-48.57633437499735, 75.27936991532934],
        [-47.683775159468986, 75.29177308180897],
        [-46.78991060824607, 75.30063916636286],
        [-45.89517371021107, 75.30596149390603],
        [-45.0, 75.30773604970747],
        [-45.0, 75.53494901291555],
        [-45.0, 75.76227192404434],
        [-45.0, 75.98970313591558],
        [-45.0, 76.21724099676472],
        [-45.95484125387219, 76.21534454348573],
        [-46.90915243299638, 76.20965680676892],
        [-47.86240522611175, 76.20018264928069],
        [-48.81407483429035, 76.1869301532171],
        [-48.751729070525975, 75.95990198028798],
        [-48.69138598645128, 75.73296427947737],
        [-48.63295073948821, 75.50611946913091],
        [-48.57633437499735, 75.27936991532934],
      ]
    ]
  },
  "properties": {}
}


def get_overlapping_dirs_for_feature(feature):
    geom = shape(feature['geometry'])

    minx, miny, maxx, maxy = geom.bounds

    # Floor to 10-degree tile grid
    start_lon = int(minx // 10) * 10
    end_lon = int(maxx // 10) * 10
    if maxx % 10 != 0: end_lon += 10

    start_lat = int(miny // 10) * 10
    end_lat = int(maxy // 10) * 10
    if maxy % 10 != 0: end_lat += 10

    intersecting_dirs = []

    for lat in range(start_lat, end_lat, 10):
        for lon in range(start_lon, end_lon, 10):
            tile = box(lon, lat, lon + 10, lat + 10)
            if tile.intersects(geom):
                lat_prefix = 'N' if lat >= 0 else 'S'
                lon_prefix = 'E' if lon >= 0 else 'W'
                dir_name = f"{lat_prefix}{abs(lat):02d}{lon_prefix}{abs(lon):03d}"
                intersecting_dirs.append(dir_name)

    return intersecting_dirs

def find_paths_for_dir_names(base_dir, dir_names):
    base = Path(base_dir)
    matches = []

    for dir_name in dir_names:
        matches.extend(p for p in base.rglob(dir_name) if p.is_dir())

    return matches

matching = find_paths_for_dir_names(destination, get_overlapping_dirs_for_feature(feature))
matching

In [ ]:
globs = [str(d / "*.parquet") for d in matching]

duckdb.sql(f"""
    SELECT count(*) FROM read_parquet({globs})
""")

In [ ]:
%pip install cql2

In [ ]:
import cql2

href = str(Path(partitioned_destination) / "**" / "*.parquet")
cql2_json = cql2.parse_text("percent_valid_pixels=100").to_json()

In [ ]:
import os.path
import humanize

destination = Path("./data/geoparquet/new")

count = 0
size = 0
for path in destination.glob("**/*.parquet"):
    count += 1
    size += os.path.getsize(path)

print(f"The {count} stac-geoparquet files are {humanize.naturalsize(size)}")
print(f"That's {100 * size / sum(sizes):.2f}% of the original size")

In [ ]:
import os.path
import humanize

destination = Path("./data/geoparquet/sentinel1")

count = 0
size = 0
for path in destination.glob("**/*.parquet"):
    count += 1
    size += os.path.getsize(path)

print(f"The {count} stac-geoparquet files are {humanize.naturalsize(size)}")
print(f"That's {100 * size / sum(sizes):.2f}% of the original size")

In [ ]:
# Debug what's happening with path parts
from pathlib import Path
from urllib.parse import urlparse

# Example path
path = "s3://its-live-data/test-space/stac-catalog/landsatOLI/v02/N30E100/2013.ndjson"

url_path = urlparse(path).path
print(f"url_path: {url_path}")

source_path = Path(url_path)
print(f"source_path.parts: {source_path.parts}")
print(f"Last 4 parts: {source_path.parts[-4:]}")

preserved_path = Path(*source_path.parts[-4:])
print(f"preserved_path: {preserved_path}")

# The fix - skip the root '/' by filtering it out
non_root_parts = [part for part in source_path.parts if part != '/']
print(f"non_root_parts: {non_root_parts}")
print(f"Last 4 non-root parts: {non_root_parts[-4:]}")

preserved_path_fixed = Path(*non_root_parts[-4:])
print(f"preserved_path_fixed: {preserved_path_fixed}")

In [ ]:


paths = [
    'landsatOLI/v02/N20E080/1987.ndjson',
    'landsatOLI/v02/N20E080/',
    'landsatOLI/v02/N20E080/README.txt',
    'landsatOLI/v02/S10W100/2003.ndjson'
]

filtered = [p for p in paths if year_file_re.match(p)]
print(filtered)